# 24. Como cada base representa endereços e estratos

O CNEFE e o BHMap descrevem endereços de formas diferentes. O CNEFE tem um registro por unidade (apartamento, loja, casa, estabelecimento) e o BHMap tem um ponto por número de porta dentro do lote. Este notebook descreve essas representações e o que delas sobra em cada conjunto de chaves. Ele responde a cinco perguntas.

1. O BHMap tem um ponto por número de porta, ou há números com mais de um ponto?
2. Como as unidades do CNEFE se condensam na chave de logradouro e número?
3. Como cada estrato usado nos resultados é identificado, e quanto a definição das vilas e favelas muda a cobertura?
4. Quem são os registros pareados, os sem número e os só do CNEFE, e quem são os pontos só do BHMap?
5. Em que tipo de lote caem os pontos só do BHMap, comparados com os pareados?

As entradas são as bases processadas do [NB01](01_ingestao.ipynb), a camada de vilas e favelas da prefeitura e o pareamento pela chave com todos os pontos do BHMap, gravado pelo [NB23](23_referencia_municipal_completa.ipynb). As tabelas saem em `outputs/referencia_completa/tables` com o prefixo `nb24_`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/config.py').exists())
sys.path.insert(0, str(ROOT))
from src import config
from src.metrics import chave_identidade, separar_logradouro_numero, conciliar_logradouros
from src.normalize import normalize_logradouro
P = config.PROCESSED_DIR
T = ROOT/'outputs/referencia_completa/tables'
T.mkdir(parents=True, exist_ok=True)

C = gpd.read_parquet(P/'cnefe_bh.parquet')
B = gpd.read_parquet(P/'bhmap_bh.parquet', columns=['IDEND', 'SIGLA_TIPO', 'NOME_LOGRA', 'NUMERO_IMO', 'NOME_BAIRR', 'NOME_REGIO', 'CEP', 'geometry'])
I = pd.read_parquet(P/'cnefe_identidade_completa.parquet')
vila = gpd.read_parquet(P/'vila_favela.parquet')
assert len(C) == 1180102 and len(B) == 757113 and I.index.equals(C.index)
lc, nc = chave_identidade(*separar_logradouro_numero(C.LOGRAD_NUM))
lb, nb = chave_identidade(B.SIGLA_TIPO.fillna('')+' '+B.NOME_LOGRA.fillna(''), B.NUMERO_IMO)
mapa = conciliar_logradouros(lc.unique(), lb.unique(), 90, 2)
lc = lc.map(lambda x: mapa.get(x, x)).astype('string[python]')
nc, lb, nb = [s.astype('string[python]') for s in (nc, lb, nb)]
C['chave'] = (lc+'|'+nc).where(lc.notna() & lc.ne('') & nc.notna())
B['chave'] = (lb+'|'+nb).where(lb.notna() & lb.ne('') & nb.notna())
assert C.chave.nunique() == 456022 and B.chave.nunique() == 613763
print(f'CNEFE: {len(C):,} registros em {C.chave.nunique():,} chaves | BHMap: {len(B):,} pontos em {B.chave.nunique():,} chaves')

CNEFE: 1,180,102 registros em 456,022 chaves | BHMap: 757,113 pontos em 613,763 chaves


## 1. Pontos do BHMap por número de porta

No BHMap, tipo, nome e número do logradouro vêm em campos separados, e cada ponto fica dentro do lote. A pergunta é se cada número de porta tem um único ponto. A tabela classifica as chaves com mais de um ponto pela posição desses pontos: na mesma coordenada, no mesmo bairro a até 100 m, no mesmo bairro a mais de 100 m ou em bairros diferentes.

In [2]:
B['x'], B['y'] = B.geometry.x.round(2), B.geometry.y.round(2)
B['bairro'] = B.NOME_BAIRR.map(lambda v: normalize_logradouro(v) if pd.notna(v) else '')
g = B.dropna(subset=['chave']).groupby('chave')
k = g.agg(pontos=('x', 'size'), xmin=('x', 'min'), xmax=('x', 'max'), ymin=('y', 'min'), ymax=('y', 'max'), bairros=('bairro', 'nunique'))
k['coords'] = B.dropna(subset=['chave']).drop_duplicates(['chave', 'x', 'y']).groupby('chave').size()
k['extensao_m'] = np.hypot(k.xmax-k.xmin, k.ymax-k.ymin)
m = k[k.pontos > 1].copy()
m['classe'] = np.select([m.coords.eq(1), m.bairros.gt(1), m.extensao_m.le(100)], ['mesma coordenada', 'bairros diferentes', 'mesmo bairro, até 100 m'], 'mesmo bairro, acima de 100 m')
varios = m.groupby('classe').agg(chaves=('pontos', 'size'), pontos=('pontos', 'sum'), extensao_mediana_m=('extensao_m', 'median')).sort_values('chaves', ascending=False)
varios.loc['total'] = [len(m), m.pontos.sum(), m.extensao_m.median()]
assert int(varios.loc['total', 'chaves']) == 95364 and int(varios.loc['bairros diferentes', 'chaves']) == 9531
varios.to_csv(T/'nb24_bhmap_chaves_varios_pontos.csv')
print(f'Chaves com um ponto: {int(k.pontos.eq(1).sum()):,} de {len(k):,}')
print(varios.round(1).to_string())

Chaves com um ponto: 518,399 de 613,763
                               chaves    pontos  extensao_mediana_m
classe                                                             
mesmo bairro, até 100 m       83978.0  208269.0                 5.3
bairros diferentes             9531.0   25946.0              9345.0
mesmo bairro, acima de 100 m   1847.0    4483.0               199.7
mesma coordenada                  8.0      16.0                 0.0
total                         95364.0  238714.0                 5.9


Das 613.763 chaves do BHMap, 518.399 têm um único ponto. Entre as 95.364 restantes, 83.978 têm pontos no mesmo bairro separados por até 100 m, 9.531 aparecem em bairros diferentes e 1.847 têm pontos a mais de 100 m no mesmo bairro. Essas relações espaciais não comprovam que os pontos descrevem o mesmo imóvel ou lote. Escolher o ponto mais próximo dentro da chave favorece distâncias menores, inclusive quando os candidatos estão próximos entre si.

A redução de registros municipais a chaves pode ser decomposta em etapas. Primeiro retiramos combinações repetidas de código de logradouro, número e letra. Depois retiramos a letra e, por fim, substituímos o código do logradouro pelo nome normalizado. Cada redução depende dessa ordem e não define uma categoria exclusiva de chaves.

In [3]:
bruto = gpd.read_file(config.RAW_DIR/'bhmap/bhmap.shp', columns=['ID_LOGRADO', 'NUMERO_IMO', 'LETRA_IMOV'])
assert len(bruto) == len(B) == 757113
com_letra = bruto.drop_duplicates(['ID_LOGRADO', 'NUMERO_IMO', 'LETRA_IMOV'])
sem_letra = bruto.drop_duplicates(['ID_LOGRADO', 'NUMERO_IMO'])
n_chaves = B.chave.nunique()
decomposicao = pd.DataFrame({
    'etapa': ['registros originais', 'combinações de código, número e letra', 'combinações de código e número', 'chaves de nome normalizado e número'],
    'unidades_restantes': [len(bruto), len(com_letra), len(sem_letra), n_chaves],
    'reducao_na_etapa': [0, len(bruto)-len(com_letra), len(com_letra)-len(sem_letra), len(sem_letra)-n_chaves]
})
assert decomposicao.reducao_na_etapa.tolist() == [0, 7796, 125184, 10370]
assert decomposicao.reducao_na_etapa.sum() == len(B)-n_chaves == 143350
decomposicao.to_csv(T/'nb24_decomposicao_condensacao_bhmap.csv', index=False)
print(decomposicao.to_string(index=False))


                                etapa  unidades_restantes  reducao_na_etapa
                  registros originais              757113                 0
combinações de código, número e letra              749317              7796
       combinações de código e número              624133            125184
  chaves de nome normalizado e número              613763             10370


Dos 757.113 registros, 7.796 repetem uma combinação de código, número e letra, restando 749.317 combinações. Descartar a letra reúne outras 125.184 combinações, restando 624.133 pares de código e número. A troca do código pelo nome normalizado reúne outras 10.370 combinações, resultando nas 613.763 chaves. A soma das três reduções é de 143.350 registros.

## 2. Unidades do CNEFE condensadas na chave

No CNEFE, logradouro e número vêm num único campo, e cada unidade de um prédio é um registro com o seu complemento. A chave descarta o complemento, de modo que todas as unidades de uma mesma porta se juntam numa chave só. A tabela mostra como isso se distribui conforme a chave tenha ou não apartamentos.

In [4]:
C['apto'] = pd.to_numeric(C.COD_TIPO_ESPECI, errors='coerce').eq(103).fillna(False).astype(bool)
C['estab'] = pd.to_numeric(C.COD_ESPECIE, errors='coerce').isin([3, 4, 5, 6, 8]).fillna(False).astype(bool)
gc = C.dropna(subset=['chave']).groupby('chave').agg(registros=('apto', 'size'), apto=('apto', 'any'), estab=('estab', 'any'))
gc['tipo'] = np.select([gc.apto.to_numpy() & gc.estab.to_numpy(), gc.apto.to_numpy(), gc.estab.to_numpy(), gc.registros.eq(1).to_numpy()],
                       ['apartamentos e estabelecimentos', 'com apartamentos', 'com estabelecimentos', 'um registro'], 'várias unidades sem apartamento')
cond = gc.groupby('tipo').agg(chaves=('registros', 'size'), registros=('registros', 'sum'), mediana=('registros', 'median'),
                              p90=('registros', lambda s: s.quantile(.9)), maximo=('registros', 'max'))
cond.loc['total'] = [len(gc), gc.registros.sum(), gc.registros.median(), gc.registros.quantile(.9), gc.registros.max()]
assert int(cond.loc['total', 'registros']) == 1157750
cond.to_csv(T/'nb24_condensacao_cnefe.csv')
com_apto = gc[gc.apto]
print(f'Chaves com um registro: {int(gc.registros.eq(1).sum()):,} de {len(gc):,}')
print(f'Chaves com apartamentos: {len(com_apto):,}, com {com_apto.registros.sum():,} registros ({100*com_apto.registros.sum()/gc.registros.sum():.1f}%)')
print(cond.sort_values('chaves', ascending=False).to_string())
C['registros_na_chave'] = C.groupby('chave').chave.transform('size')
compartilhada = C[C.chave.notna() & C.registros_na_chave.gt(1)]
complemento = compartilhada.COMPLEMENTO.fillna('').str.strip().ne('')
partilha = pd.Series({'registros_com_numero': int(C.chave.notna().sum()), 'registros_em_chave_compartilhada': len(compartilhada),
                      'pct_em_chave_compartilhada': 100*len(compartilhada)/C.chave.notna().sum(),
                      'pct_compartilhada_com_complemento': 100*complemento.mean()})
partilha.to_csv(T/'nb24_chave_compartilhada.csv')
print(partilha.round(1).to_string())
multi = gc[gc.registros.gt(1)].copy()
tipo_porta = np.where(multi.apto.to_numpy(), 'chave compartilhada com apartamentos', 'chave compartilhada sem apartamentos')
portas = multi.assign(tipo=tipo_porta).groupby('tipo').agg(portas=('registros', 'size'), registros=('registros', 'sum'), mediana=('registros', 'median'))
portas['pct_registros_com_numero'] = 100*portas.registros/gc.registros.sum()
portas.loc['total'] = [len(multi), multi.registros.sum(), multi.registros.median(), 100*multi.registros.sum()/gc.registros.sum()]
portas.to_csv(T/'nb24_portas_compartilhadas.csv')
print(portas.round(1).to_string())
# Distingo complemento preenchido de complementos distintos dentro da chave.
# Normalização conservadora: caixa, espaços e Unicode, sem remover números.
comp = compartilhada.COMPLEMENTO.fillna('').astype(str).str.normalize('NFKC').str.upper().str.replace(r'\s+', ' ', regex=True).str.strip()
aud = pd.DataFrame({'chave': compartilhada.chave, 'complemento': comp})
ng = aud[comp.ne('')].groupby('chave').complemento.nunique()
com_distintos = aud.chave.map(ng).fillna(0).ge(2)
repetido = comp.ne('') & aud.duplicated(['chave', 'complemento'], keep=False)
resumo_complemento = pd.Series({
    'registros_com_numero': int(C.chave.notna().sum()),
    'registros_em_chave_compartilhada': len(aud),
    'chaves_compartilhadas': aud.chave.nunique(),
    'registros_com_complemento_preenchido': int(comp.ne('').sum()),
    'chaves_com_dois_complementos_distintos': int(ng.ge(2).sum()),
    'registros_em_chaves_com_dois_complementos_distintos': int(com_distintos.sum()),
    'pct_compartilhados_em_chaves_com_dois_complementos_distintos': 100*com_distintos.mean(),
    'pct_numerados_em_chaves_com_dois_complementos_distintos': 100*com_distintos.sum()/C.chave.notna().sum(),
    'registros_com_complemento_repetido_na_chave': int(repetido.sum()),
    'registros_sem_complemento_na_chave_compartilhada': int(comp.eq('').sum()),
})
resumo_complemento.to_csv(T/'nb24_complementos_distintos.csv')
print(resumo_complemento.round(3).to_string())


Chaves com um registro: 257,058 de 456,022
Chaves com apartamentos: 37,073, com 432,046 registros (37.3%)
                                   chaves  registros  mediana   p90  maximo
tipo                                                                       
total                            456022.0  1157750.0      1.0   4.0  1221.0
um registro                      202347.0   202347.0      1.0   1.0     1.0
várias unidades sem apartamento  117880.0   328273.0      2.0   4.0   260.0
com estabelecimentos              98722.0   195084.0      1.0   4.0    92.0
com apartamentos                  33375.0   389369.0      6.0  21.0  1221.0
apartamentos e estabelecimentos    3698.0    42677.0      5.0  19.0   771.0
registros_com_numero                 1157750.0
registros_em_chave_compartilhada      900692.0
pct_em_chave_compartilhada                77.8
pct_compartilhada_com_complemento         73.1
                                        portas  registros  mediana  pct_registros_com_numero
tipo 

Os 1.157.750 registros com número formam 456.022 chaves. Destes registros, 900.692 (77,8%) compartilham a chave com outro registro. As chaves compartilhadas que contêm apartamentos reúnem 37,1% dos registros numerados, e as demais reúnem 40,7%. Esses percentuais descrevem grupos textuais, não prédios fisicamente confirmados. O complemento está preenchido em 73,1% dos registros com chave compartilhada. A tabela adicional separa esse preenchimento da presença de pelo menos dois complementos distintos na mesma chave e identifica complementos repetidos. A mesma chave pode reunir entradas diferentes, modificadores descartados ou logradouros homônimos. A cobertura continua contada por registro, enquanto a sobreposição dos cadastros usa a chave, sem fundir as coordenadas usadas nas distâncias.

## 3. Como os estratos são identificados

Nenhuma das duas bases marca vila, favela, comércio ou residência da mesma forma. Os estratos usados nos resultados vêm de três fontes.

- **Apartamentos, estabelecimentos e demais tipos:** atributos do CNEFE. Apartamento é o tipo de domicílio 103. Estabelecimento é a espécie de ensino, saúde, religioso, agropecuário ou outras finalidades (3, 4, 5, 6 e 8). Os demais são domicílios que não são apartamentos e edificações em construção. O BHMap não distingue unidades nem uso.
- **Comércio no lado municipal:** como o BHMap não tem uso, a referência comercial é a camada de atividade econômica da prefeitura, seção G da CNAE.
- **Vilas e favelas:** nenhuma das bases tem esse atributo. O estrato vem da camada de vilas e favelas da prefeitura, por ponto em polígono sobre a coordenada do CNEFE. A camada reúne dois tipos de assentamento, e a tabela mede a cobertura de cada um.

In [5]:
seg = pd.Series(np.select([C.apto.to_numpy(), C.estab.to_numpy()], ['apartamentos', 'estabelecimentos'], 'demais tipos'), index=C.index)
assert seg.value_counts().to_dict() == {'demais tipos': 642844, 'apartamentos': 403623, 'estabelecimentos': 133635}
par = I.pareado_identidade
cauda = par & I.dist_identidade.gt(100)
linhas = []
for tipo in ['VILA_FAVELA', 'LOTEAMENTO_SOCIAL']:
    sub = vila[vila.TIPO_ASSENTAMENTO.eq(tipo)]
    dentro = pd.Series(shapely.within(C.geometry.to_numpy(), sub.geometry.union_all()), index=C.index)
    linhas.append({'recorte': tipo, 'poligonos': len(sub), 'registros': int(dentro.sum()), 'cobertura_pct': 100*par[dentro].mean(),
                   'cauda_pct_dos_pares': 100*cauda[dentro].sum()/par[dentro].sum()})
em_vila = pd.Series(shapely.within(C.geometry.to_numpy(), vila.geometry.union_all()), index=C.index)
for nome, mask in [('camada inteira', em_vila), ('fora da camada', ~em_vila)]:
    linhas.append({'recorte': nome, 'poligonos': len(vila) if nome == 'camada inteira' else 0, 'registros': int(mask.sum()),
                   'cobertura_pct': 100*par[mask].mean(), 'cauda_pct_dos_pares': 100*cauda[mask].sum()/par[mask].sum()})
estratos_vila = pd.DataFrame(linhas).set_index('recorte')
assert int(estratos_vila.loc['camada inteira', 'registros']) == 128149
estratos_vila.to_csv(T/'nb24_estratos_vila.csv')
print(estratos_vila.round(2).to_string())

                   poligonos  registros  cobertura_pct  cauda_pct_dos_pares
recorte                                                                    
VILA_FAVELA              192      94188          69.99                 6.32
LOTEAMENTO_SOCIAL         26      33961          78.83                 6.76
camada inteira           218     128149          72.33                 6.45
fora da camada             0    1051953          92.15                 2.46


A camada da prefeitura tem 192 vilas e favelas e 26 loteamentos sociais. As vilas e favelas têm 94.188 registros do CNEFE e cobertura de 70,0%. Os loteamentos sociais têm 33.961 registros e cobertura de 78,8%. Juntos, somam os 128.149 registros e a cobertura de 72,3% usados nos resultados. Restringir o estrato às vilas e favelas baixaria a cobertura do estrato em pouco mais de dois pontos e não mudaria o contraste com o resto da cidade (92,1%), nem a proporção de pares acima de 100 m, que fica entre 6,3% e 6,8% nos dois tipos, contra 2,5% fora da camada. O estrato reúne, portanto, assentamentos de interesse social, e não só vilas e favelas.

## 4. Quem está em cada conjunto

Os registros do CNEFE se dividem em três grupos: os que têm a chave também no BHMap e recebem par, os que não têm número e não formam chave, e os que têm uma chave que o BHMap não tem. Do lado municipal, os pontos só do BHMap são os que têm uma chave que o CNEFE não tem. As tabelas descrevem cada grupo por tipo, espécie, nível da coordenada e vila.

In [6]:
comuns = set(C.chave.dropna()) & set(B.chave.dropna())
C['conjunto'] = np.select([C.chave.isna().to_numpy(), C.chave.isin(comuns).to_numpy()], ['sem número', 'pareado'], 'só CNEFE')
assert C.conjunto.value_counts().to_dict() == {'pareado': 1062044, 'só CNEFE': 95706, 'sem número': 22352}
assert C.conjunto.eq('pareado').equals(I.pareado_identidade)
esp = pd.to_numeric(C.COD_ESPECIE).map(config.ESPECIE_MAP)
perfil = pd.concat({'tipo': pd.crosstab(C.conjunto, seg, normalize='index'),
                    'nivel_coordenada': pd.crosstab(C.conjunto, C.NV_GEO_COORD, normalize='index'),
                    'vila': pd.crosstab(C.conjunto, em_vila.map({True: 'em vila', False: 'fora'}), normalize='index')}, axis=1)*100
perfil['registros'] = C.conjunto.value_counts()
perfil.to_csv(T/'nb24_perfil_conjuntos_cnefe.csv')
pd.crosstab(C.conjunto, esp).to_csv(T/'nb24_especie_conjuntos_cnefe.csv')
print(perfil.round(1).to_string())
print(pd.crosstab(C.conjunto, esp).to_string())

                   tipo                               nivel_coordenada                         vila       registros
           apartamentos demais tipos estabelecimentos                1     2    3    4    6 em vila  fora          
conjunto                                                                                                           
pareado            36.3         52.8             10.9             83.4  15.6  0.7  0.3  0.0     8.7  91.3   1062044
sem número          2.9         67.6             29.5             99.1   0.1  0.2  0.6  0.0    22.5  77.5     22352
só CNEFE           18.4         69.4             12.2             90.0   8.8  0.8  0.4  0.0    31.8  68.2     95706
COD_ESPECIE  Domicílio Coletivo  Domicílio Particular  Em Construção  Estab. Agropecuário  Estab. Ensino  Estab. Religioso  Estab. Saúde  Outras Finalidades
conjunto                                                                                                                                           

Os registros pareados têm 36% de apartamentos, 83% de coordenadas coletadas em campo e 9% de registros nas áreas de assentamentos da camada municipal. Entre os 95.706 registros com chave apenas no CNEFE, 18% são apartamentos, 90% têm coordenada de campo e 32% estão nessas áreas, contra 10,9% da base completa. Há sobrerrepresentação dessas áreas, mas a maioria dos registros exclusivos está fora delas. A coleta em campo descreve a origem da coordenada e não comprova sua acurácia. A ausência de par é definida pela chave adotada e não demonstra omissão física do endereço.

In [7]:
B['em_vila'] = shapely.within(B.geometry.to_numpy(), vila.geometry.union_all())
so_b = B[B.chave.notna() & ~B.chave.isin(set(C.chave.dropna()))]
assert so_b.chave.nunique() == 208967
print(f'Pontos só do BHMap: {len(so_b):,} em {so_b.chave.nunique():,} chaves | em vila: {100*so_b.em_vila.mean():.1f}% contra {100*B.em_vila.mean():.1f}% do BHMap inteiro')


Pontos só do BHMap: 226,432 em 208,967 chaves | em vila: 22.9% contra 16.5% do BHMap inteiro


Os 226.432 pontos com chave apenas no BHMap formam 208.967 chaves. Nas áreas da camada municipal de assentamentos estão 22,9% desses pontos, contra 16,5% de todos os pontos do BHMap. As associações a outro número na mesma rua e lote, medidas no NB23, são compatíveis com diferenças de numeração ou representação, mas não comprovam que o Censo registrou o mesmo imóvel de outra forma.

## 5. Uso do lote onde caem os pontos do BHMap

Uma hipótese para os pontos só do BHMap é que o cadastro municipal numera lugares onde o Censo não tem unidade para listar, como lotes vagos ou frentes não residenciais. A camada de tipologia de uso e ocupação dos lotes da prefeitura permite testar isso. Cada ponto do BHMap recebe a tipologia do lote que o contém, e a distribuição dos pontos pareados é comparada com a dos pontos só do BHMap. O mesmo teste não serve para o CNEFE, porque a coordenada do Censo marca a entrada do imóvel, em geral na calçada, fora do polígono do lote.

In [8]:
uso = gpd.read_file(config.TIPOLOGIA_USO_SHP, columns=['NULOTCTM', 'TIPOLOGIA_', 'MORFOLOGIA'])
assert uso.crs == B.crs
ju = gpd.sjoin(B[['geometry']], uso[['TIPOLOGIA_', 'MORFOLOGIA', 'geometry']], how='left', predicate='within')
ambiguos_uso = int(ju.index[ju.index.duplicated(keep=False)].nunique())
print('Pontos contidos em mais de um polígono de uso:', ambiguos_uso)
ju = ju[~ju.index.duplicated()].reindex(B.index)
B['uso_lote'] = ju.TIPOLOGIA_.fillna('fora de lote')
B['grupo'] = np.where(B.chave.isin(set(C.chave.dropna())), 'pareado', 'só BHMap')
uso_grupo = pd.crosstab(B.grupo, B.uso_lote, normalize='index')*100
uso_grupo['pontos'] = B.grupo.value_counts()
uso_grupo.to_csv(T/'nb24_uso_lote_bhmap.csv')
print(uso_grupo.round(1).to_string())


Pontos contidos em mais de um polígono de uso: 44
uso_lote  LOTE VAGO  MISTO  NAO RESIDENCIAL  PARQUE  RESIDENCIAL  SEM INFORMACAO  fora de lote  pontos
grupo                                                                                                 
pareado         2.6   13.0              7.5     0.0         61.4            15.0           0.5  530681
só BHMap       10.0   10.6             12.5     0.2         37.0            28.4           1.3  226432


Os pontos só do BHMap caem em lote vago quatro vezes mais que os pareados (10,0% contra 2,6%) e com mais frequência em lote não residencial (12,5% contra 7,5%). A tipologia também falta mais nos pontos só do BHMap (28,4% sem informação, contra 15,0%). O resultado é compatível com diferenças de escopo entre enumeração censitária e cadastro municipal. Não comprova a ocupação do lote na data do Censo nem adjudica a ausência de unidade. Os pontos sem tipologia e os lotes residenciais continuam sendo a maior parte. Para eles, a associação a outro número no mesmo lote é compatível com diferenças de numeração, sem comprová-las.

## Conclusão

A comparação exige distinguir registros de unidades, pontos municipais e chaves textuais. A chave aproxima as granularidades para contar sobreposição, mas não confirma identidade física. Complementos preenchidos e complementos distintos são medidas diferentes. Os estratos funcionais vêm do tipo e da espécie do CNEFE, enquanto a atividade econômica e as áreas de assentamentos vêm de camadas municipais. A área territorial analisada reúne 192 vilas e favelas e 26 loteamentos sociais. A composição dos conjuntos exclusivos aponta diferenças de representação e possíveis lacunas, cuja causa exige verificação independente. Os pontos só do BHMap caem em lote vago quatro vezes mais que os pareados, o que é compatível com diferenças de escopo, sem comprovar ausência de unidade na data do Censo.

O [NB25](25_setor_censitario.ipynb) compara os pares pelo setor censitário e mede a dependência espacial do deslocamento.